# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amitkumar15x/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [5]:
!pip install duckdb huggingface_hub pandas pyarrow


In [7]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token)
print("Login successful!")

Login successful!


In [8]:
import duckdb

con = duckdb.connect()

con.sql("""
INSTALL httpfs;
LOAD httpfs;
""")

print("DuckDB ready!")


DuckDB ready!


In [9]:
con.sql("""
SELECT 1;
""").show()

┌───────┐
│   1   │
│ int32 │
├───────┤
│     1 │
└───────┘



In [10]:
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected!")

Connected!


In [11]:
con.sql(f"""
DESCRIBE
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

### Unit of analysis + time window

- One row represents one content item (`content_hash_id`) for one client (`client_hash_id`) on one reporting date (`report_date`).
- The analysis uses the month `2026-03`.
- This contract is designed to build features that are available at the decision time and avoids using future information.

In [12]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '-' || content_hash_id || '-' || CAST(report_date AS VARCHAR))
        AS unique_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────┐
│ total_rows │ unique_rows │
│   int64    │    int64    │
├────────────┼─────────────┤
│    9841378 │     9841378 │
└────────────┴─────────────┘



### Fields

**Features**
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- sessions_ai
- scroll_events

**Label / Proxy**
- Future search performance or ranking improvement (defined later in the modeling workflow).

**Context**
- client_hash_id
- content_hash_id
- report_date
- month

**Excluded**
- Any information that would only be known after the prediction date (future observations or label-derived values) to prevent data leakage.

In [15]:
con.sql(f"""
SELECT COUNT(*)
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      9841378 │
└──────────────┘



In [16]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT (
            client_hash_id,
            content_hash_id,
            report_date
        )
    ) AS unique_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────┐
│ total_rows │ unique_rows │
│   int64    │    int64    │
├────────────┼─────────────┤
│    9841378 │     9841378 │
└────────────┴─────────────┘



In [17]:
con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS duplicate_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬────────────────┐
│ client_hash_id │ content_hash_id │ report_date │ duplicate_rows │
│    varchar     │     varchar     │    date     │     int64      │
├────────────────┴─────────────────┴─────────────┴────────────────┤
│                             0 rows                              │
└─────────────────────────────────────────────────────────────────┘



In [18]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").show()

┌────────────┬────────────┬────────────┐
│ total_rows │ first_day  │  last_day  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘



In [19]:
con.sql(f"""
SELECT
    COUNT(*) AS gsc_available_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│ gsc_available_rows │
│       int64        │
├────────────────────┤
│            3611061 │
└────────────────────┘



## Unit of analysis + time window

- One row represents one content item (`content_hash_id`) for one client (`client_hash_id`) on one reporting date (`report_date`).
- The analysis uses data from the month **2026-03**.
- The grain was verified by checking for duplicate `(client_hash_id, content_hash_id, report_date)` combinations, and no duplicates were found.

## Fields

### Features
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- sessions_ai
- scroll_events

### Label / Proxy
- Future search performance (to be defined during the modeling stage).

### Context
- client_hash_id
- content_hash_id
- report_date
- month

### Excluded
- Future observations and any label-derived columns are excluded because they would introduce data leakage.

## Verification

The contract was verified with three SQL queries.

### 1. Grain
No duplicate `(client_hash_id, content_hash_id, report_date)` combinations were found, confirming that one row represents one content item for one client on one reporting date.

### 2. Row count and time window
For month **2026-03**, the warehouse contains **9,841,378 rows** covering **2026-03-01** through **2026-03-31**.

### 3. Availability
Filtering with `gsc_data_available IS TRUE` returns **3,611,061 rows**, confirming that Search Console data availability varies across the dataset.

## Data limits

- The dataset is anonymized and does not contain client names or URLs.
- Search Console and GA4 availability differ across clients and dates.
- The analysis is observational and supports decision-making; it does not establish causation.
- Only data available at the decision time should be used to avoid leakage from future information.